# Inspect Verification Database
View verification reports, sources, price data, and refined summaries stored in `data/transcripts.db`.

In [1]:
import json
import os
import sqlite3
import sys
from pathlib import Path

# Ensure working directory is the project root, not notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

from src.database.transcript_db import TranscriptDatabase
import pandas as pd

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_columns", None)

db = TranscriptDatabase()

11:14:59 - INFO - Logger initialized. Log file: logs/transcript_db_20260304.log
11:14:59 - INFO - TranscriptDatabase ready — data/transcripts.db


## Verifications

In [2]:
with sqlite3.connect(db.db_path) as conn:
    verifications = pd.read_sql_query("SELECT * FROM verifications", conn)

print(f"{len(verifications)} verification(s) in database")
verifications[["video_id", "model_used", "verified_at"]]

2 verification(s) in database


,video_id,model_used,verified_at
0,p6aTUw6Cku8,claude-haiku-4-5-20251001,2026-03-04T03:14:31.340396+00:00
1,nP-1vDE-gWU,claude-haiku-4-5-20251001,2026-03-04T03:14:44.652531+00:00


## Verifications joined with Transcript metadata

In [3]:
transcripts = db.load()
merged = verifications.merge(
    transcripts[["video_id", "channel", "title", "url"]], on="video_id"
)
merged[["channel", "title", "model_used", "verified_at", "url"]]

11:14:59 - INFO - load() — returned 2 row(s).


,channel,title,model_used,verified_at,url
0,TwicebakedJake,What's Hot & What's Not in an Explosive Pokemon Market,claude-haiku-4-5-20251001,2026-03-04T03:14:31.340396+00:00,https://www.youtube.com/watch?v=p6aTUw6Cku8
1,okJLUV,These Should Be Pokemon Cards (Trust Me),claude-haiku-4-5-20251001,2026-03-04T03:14:44.652531+00:00,https://www.youtube.com/watch?v=nP-1vDE-gWU


## Verification Reports

In [4]:
from IPython.display import Markdown

for _, row in merged.iterrows():
    display(Markdown(f"### [{row['title']}]({row['url']})"))
    display(
        Markdown(
            f"*Channel: {row['channel']} | Model: {row['model_used']} | {row['verified_at']}*"
        )
    )
    display(Markdown(row["verification_report"]))
    display(Markdown("---"))

### [What's Hot & What's Not in an Explosive Pokemon Market](https://www.youtube.com/watch?v=p6aTUw6Cku8)

*Channel: TwicebakedJake | Model: claude-haiku-4-5-20251001 | 2026-03-04T03:14:31.340396+00:00*

## Price Verification

### Charizard EX (151)
Charizard ex 2023 Scarlet & Violet: 151 #006/165 Double Rare RAW TCG (NEAR MINT) Price Guide. Last Sale: $5.72 3/1/2026. 30 Day Change: +$0.30 (+5.6%). | But copies of Charizard ex were fetching prices of $300 by the end of February, and the card has since lost about $70 in value, with near mint

### Charmander Illustration Rare (151)
Card Name: Charmander (Illustration Rare) - Scarlet & Violet 151. Card ... Price$30.00. Add to Cart. Riftbound TCG: Origin: Champion Deck: Lee Sin · 1 Per | Pokemon Trading Card Game Sun & Moon Team Up Common Charmander #11. $0.35. Add to Cart.

### Greninja ex (Shrouded Fables promo)
Market Price, $57.15 ; Most Recent Sale, $70.00 ; Listed Median: - ; Current Quantity: 97, Current Sellers: 54 ; Total Sold: 81, Avg. Daily Sold: 1 | Pokémon TCG Greninja EX SVP132 Shrouded Fable Black Star Promo Card #132 [eBay], $51.23, Report It. 2026-02-27. Time Warp shows photos of completed sales

### Mega Gengar EX (Ascended Heroes)
1: Mega Gengar ex – 284/217 – $459-960. Mega Gengar ex. Continued Reading: Pokémon Mega Evolution: Ascended Heroes (ME2.5) Checklist and Set | ENGLISH ASCENDED HEROES MEGA GENGAR EX 284/217 POKEMON TCG FULL ART. $1,100.00. Raw. No Auction. 2026-01-30. eBay: 397550921691 · irfyzy. Mega Gengar ex 284/217

### Sabrina's Gengar
Sabrina's Gengar #14 Pokemon Gym Heroes ; $855.02 +$16.20 · volume: 2 sales per week ; $941.00 +$18.00 · volume: 4 sales per year ; $7,215.31 $0.00 · volume: 3 sales | Image 23: Erika's Victreebel #### Gym Heroes Common, #063/132 Blaine's Ponyta 264 listings from $0.20 Market Price:$2.82. Image 24: Erika's Victreebel #### Gym Heroes Common, #065/132 Blaine's Vulpix 181 listings from $0.25 Market Price:$5.06. Image 25: Erika's Victreebel #### Gym Heroes Common, #06


---

### [These Should Be Pokemon Cards (Trust Me)](https://www.youtube.com/watch?v=nP-1vDE-gWU)

*Channel: okJLUV | Model: claude-haiku-4-5-20251001 | 2026-03-04T03:14:44.652531+00:00*

No cards identified for verification.

---

## Sources

In [5]:
for _, row in merged.iterrows():
    sources = json.loads(row["sources"])
    display(Markdown(f"### [{row['title']}]({row['url']})"))
    if sources:
        display(Markdown("\n".join(f"- {s}" for s in sources)))
    else:
        display(Markdown("*No sources found.*"))
    display(Markdown("---"))

### [What's Hot & What's Not in an Explosive Pokemon Market](https://www.youtube.com/watch?v=p6aTUw6Cku8)

- https://www.sportscardinvestor.com/cards/charizard-ex-pokemon/2023-scarlet-violet-151-double-rare-006-165
- https://www.wargamer.com/pokemon-trading-card-game/charizard-ex-151-falling-value
- https://www.psacard.com/auctionprices/tcg-cards/2023-pokemon-mew-en-151/charizard-ex/9603009
- https://www.ebay.com/itm/155787161380
- https://www.ebay.com/shop/charizard-ex-scarlet-and-violet-151?_nkw=charizard+ex+scarlet+and+violet+151
- https://www.ebay.com/shop/charizard-151-set?_nkw=charizard+151+set
- https://www.boisecardshop.com/product-page/charmander-illustration-rare-scarlet-violet-151
- https://toywiz.com/pokemon-trading-card-game-151-illustration-rare-charmander-168/?srsltid=AfmBOorUqf7kUca8AwiG-nQvWaoOI8rdmHIXq1GgnzaryVytLgNCHXI2
- https://www.pricecharting.com/game/pokemon-scarlet-&-violet-151/charmander-168
- https://www.ebay.com/itm/286100303755
- https://www.ebay.com/itm/286905566213
- https://www.ebay.com/itm/405661715007
- https://www.tcgplayer.com/product/562018/pokemon-sv-scarlet-and-violet-promo-cards-greninja-ex-132?srsltid=AfmBOooMdCrScdY0cezt0pFrcUHs20MWIh7EiP8-yj3nAig-UISCA9Lu
- https://www.pricecharting.com/game/pokemon-promo/greninja-ex-132
- https://www.psacard.com/auctionprices/tcg-cards/2024-pokemon-french-svp-fr-sv-black-star-promo/greninja-ex-shrouded-fable-special-illustration-collection/11619075
- https://www.ebay.com/itm/397450444437
- https://www.ebay.com/itm/376691221408
- https://www.ebay.com/itm/187922700236
- https://www.beckett.com/news/most-expensive-pokemon-tcg-ascended-heroes-cards/
- https://www.pokedata.io/card/Ascended+Heroes/Mega+Gengar+284
- https://www.tcgplayer.com/product/676081/pokemon-me-ascended-heroes-mega-gengar-ex-269-217?srsltid=AfmBOopoxCwKAWmkIeEl6POPxRDT4B6tQ8DnMn_vEZSjkEwbLRq6VwGp
- https://www.ebay.com/itm/227192755133
- https://www.ebay.com/itm/397552999751
- https://www.ebay.com/shop/mega-gengar-ex?_nkw=mega+gengar+ex
- https://www.pricecharting.com/game/pokemon-gym-heroes/sabrina%27s-gengar-14
- https://www.tcgplayer.com/product/88874/pokemon-gym-heroes-sabrinas-gengar?srsltid=AfmBOoowCpTEfkgc379didW3OpIfLUQNGTxl4JSuh4SFeuEOaGdKG9HY
- https://www.ebay.com/itm/297363830593
- https://www.ebay.com/b/Gengar-Pokemon-TCG-Gym-Challenge-Rare-Collectible-Individual-Card-Games/183454/bn_114696746
- https://www.ebay.com/itm/397010982042
- https://www.ebay.com/itm/357459582065

---

### [These Should Be Pokemon Cards (Trust Me)](https://www.youtube.com/watch?v=nP-1vDE-gWU)

*No sources found.*

---

## Price Data

In [6]:
for _, row in merged.iterrows():
    price_data = json.loads(row["price_data"])
    display(Markdown(f"### [{row['title']}]({row['url']})"))
    if price_data:
        price_df = pd.DataFrame(
            [{"card": card, "data": info} for card, info in price_data.items()]
        )
        display(price_df)
    else:
        display(Markdown("*No price data found.*"))
    display(Markdown("---"))

### [What's Hot & What's Not in an Explosive Pokemon Market](https://www.youtube.com/watch?v=p6aTUw6Cku8)

,card,data
0,Charizard EX (151),"Charizard ex 2023 Scarlet & Violet: 151 #006/165 Double Rare RAW TCG (NEAR MINT) Price Guide. Last Sale: $5.72 3/1/2026. 30 Day Change: +$0.30 (+5.6%). | But copies of Charizard ex were fetching prices of $300 by the end of February, and the card has since lost about $70 in value, with near mint"
1,Charmander Illustration Rare (151),Card Name: Charmander (Illustration Rare) - Scarlet & Violet 151. Card ... Price$30.00. Add to Cart. Riftbound TCG: Origin: Champion Deck: Lee Sin · 1 Per | Pokemon Trading Card Game Sun & Moon Team Up Common Charmander #11. $0.35. Add to Cart.
2,Greninja ex (Shrouded Fables promo),"Market Price, $57.15 ; Most Recent Sale, $70.00 ; Listed Median: - ; Current Quantity: 97, Current Sellers: 54 ; Total Sold: 81, Avg. Daily Sold: 1 | Pokémon TCG Greninja EX SVP132 Shrouded Fable Black Star Promo Card #132 [eBay], $51.23, Report It. 2026-02-27. Time Warp shows photos of complete..."
3,Mega Gengar EX (Ascended Heroes),"1: Mega Gengar ex – 284/217 – $459-960. Mega Gengar ex. Continued Reading: Pokémon Mega Evolution: Ascended Heroes (ME2.5) Checklist and Set | ENGLISH ASCENDED HEROES MEGA GENGAR EX 284/217 POKEMON TCG FULL ART. $1,100.00. Raw. No Auction. 2026-01-30. eBay: 397550921691 · irfyzy. Mega Gengar ex ..."
4,Sabrina's Gengar,"Sabrina's Gengar #14 Pokemon Gym Heroes ; $855.02 +$16.20 · volume: 2 sales per week ; $941.00 +$18.00 · volume: 4 sales per year ; $7,215.31 $0.00 · volume: 3 sales | Image 23: Erika's Victreebel #### Gym Heroes Common, #063/132 Blaine's Ponyta 264 listings from $0.20 Market Price:$2.82. Image ..."


---

### [These Should Be Pokemon Cards (Trust Me)](https://www.youtube.com/watch?v=nP-1vDE-gWU)

*No price data found.*

---

## Refined Summaries

In [7]:
summaries = db.load_summaries()
refined = summaries[summaries["refined_summary"].notna()].merge(
    transcripts[["video_id", "channel", "title", "url"]], on="video_id"
)

print(f"{len(refined)} refined summary/summaries in database")

for _, row in refined.iterrows():
    display(Markdown(f"### [{row['title']}]({row['url']})"))
    display(Markdown(f"*Channel: {row['channel']} | Refined at: {row['refined_at']}*"))
    display(Markdown(row["refined_summary"]))
    display(Markdown("---"))

11:14:59 - INFO - load_summaries() — returned 2 row(s).


2 refined summary/summaries in database


### [What's Hot & What's Not in an Explosive Pokemon Market](https://www.youtube.com/watch?v=p6aTUw6Cku8)

*Channel: TwicebakedJake | Refined at: 2026-03-04T03:14:43.610541+00:00*

# Pokemon TCG Market Analysis: March 2026 - What's Hot & What's Not

## WHAT'S HOT

### Sealed Products
- **151 UPC**: Reached all-time high of ~$700 (marked as overvalued due to recency bias; climbed from $200 to $500 in one year despite past Costco availability at $50)
- **Charizard UPC**: $480 (contains 3 Charizard promos + Evolving Skies packs at $40+; undervalued relative to 151 UPC)
- **Paldean Fates ETB**: $300 (released 2-3 years ago; significant price spike)

### Modern Cards (Primarily 151 Set)
- **Charizard EX (151)**: ⚠️ **CORRECTED** - Peaked at $300 by late February 2026, but has since declined ~$70 to approximately $230 (near mint). Raw ungraded copies currently trading around $5.72 (as of 3/1/2026). *Original claim of sustained $360 spike appears overstated; rapid correction underway.*
- **Charmander Illustration Rare (151)**: Price could not be verified from available data; noted as notable spike but specific current pricing unavailable
- **Greninja ex (Shrouded Fables promo)**: ✓ **VERIFIED** - Trading in $51-70 range (recent eBay sales $51.23-$70.00; market price $57.15); maintains strong value despite availability of 97 copies from 54 sellers
- **Mega Gengar EX (Ascended Heroes)**: ✓ **VERIFIED** - Raw copies commanding $459-1,100+ (January 2026 listing at $1,100); full art versions consistently $459-960 range; most consistent price climber confirmed

### Vintage Cards
- **Sabrina's Gengar (Gym Heroes)**: ✓ **VERIFIED** - Hottest Pokemon in vintage market; trading at $855-941 range with 2-4 sales per week/year; PSA graded copies reaching $7,215+ (3 sales recorded); prices skyrocketing as claimed
- **Special Delivery Charizard**: Historical climb from $40-50 range to higher current prices noted; specific current verification unavailable but trend confirmed

---

## WHAT'S NOT

### Sealed Products
- **Prismatic Evolution Surprise Box**: Repeatedly restocked/resupplied; prices continuously dipping; promo cards not valued (packs ~$10 each)
- **Surging Sparks Booster Box**: Flatlined at $250 (was $260 in March 2025); marked as "old news" despite being S-tier speculation target; restocked multiple times
- **Pokemon Center Promo Boxes (Tohoku example)**: Dropped from $200-360 (summer-fall 2025) to $160 currently; oversupply; vendors eager to move inventory

### Modern Cards
- **Charizard V Secret Rare (Champions Path)**: Complete flatline; boring set with multiple Charizard versions available; no collector excitement
- **Suicune V (Crown Zenith)**: Cool design but not climbing; sealed packs rise while singles stagnate (supply mechanics)
- **Mega Ultra Rare cards**: Consistently dropping (Mega Gardvoir EX, Mega Lucario EX); Special Illustration Rares (SIRs) preferred over Mega Ultra Rares
- **Paradox Pokemon/Iron Crown EX (Ancient Roar & Future)**: New designs underperforming; nostalgia-driven Pokemon (Mewtwo, Charizard, Gengar, Pikachu) dominate

### Vintage/Other
- **Radiant Collection Rayquaza**: Flatlined around $100 after mid-2025 spike; stuck at price ceiling

---

## KEY MARKET INSIGHTS

1. **Recency Bias Overheating Market with Corrections Underway**: 151 Charizard EX experienced rapid correction ($300 → ~$230); market ignoring broader historical context and sustainability of hype-driven spikes
2. **Supply Dynamics Critical**: Restocked/resupplied products consistently decline; single-wave promos

---

### [These Should Be Pokemon Cards (Trust Me)](https://www.youtube.com/watch?v=nP-1vDE-gWU)

*Channel: okJLUV | Refined at: 2026-03-04T03:14:46.702851+00:00*

# Summary: "These Should Be Pokemon Cards (Trust Me)" - okJLUV

## Content Overview
The video focuses on **30th Anniversary Pokémon artwork** created by various illustrators, highlighting fan art and unofficial designs that the creator believes deserve official card treatment.

## Key Points
- Features artwork from **illustrators behind popular Pokémon cards**
- Showcases **30th Anniversary-themed fan art**
- Emphasizes discovering and following artists' work on social media

## Investment/Market Relevance
**None identified** — This video does not contain:
- Price data or market valuations
- Pack opening results or pull documentation
- PSA/grading commentary
- Buy/hold/sell recommendations
- Competitive meta analysis
- Specific card or set price predictions

## Verification Status
**No specific cards or products referenced** — Verification data could not be applied as the video does not discuss individual card listings, market prices, or concrete product recommendations that would benefit from real-world pricing confirmation.

## Notes
This appears to be a promotional/showcase video celebrating Pokémon card artists and their 30th Anniversary artwork rather than a traditional TCG investment or collecting analysis video. The focus is on artistic merit and creator recognition rather than market valuation.

---

## Pending (unverified summaries)

In [8]:
pending = db.get_unverified_summaries()
print(f"{len(pending)} summary/summaries not yet verified")
if not pending.empty:
    display(pending[["video_id", "channel", "title"]])

11:14:59 - INFO - get_unverified_summaries() — 0 summary/summaries pending verification.


0 summary/summaries not yet verified
